# Interpretabilidad — Attention maps y análisis de tokens

Análisis de interpretabilidad sobre los mejores modelos de V3:

1. **Cross-attention heatmaps (T5):** qué partes del artículo "mira" el
   modelo al generar cada token del resumen.
2. **Token importance (T5):** qué tokens del input acumulan más atención a
   lo largo de toda la generación — identifica las "zonas clave" del artículo.
3. **Comparativa cualitativa:** cómo T5 y Qwen3 priorizan información
   distinta del mismo artículo.

**Nota técnica:** la extracción de attention maps se realiza con generación
greedy (`num_beams=1`) para obtener mapas limpios. Beam search produce
distribuciones de atención más difusas que son difíciles de interpretar.

In [ ]:
# Setup
import sys
import os
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import torch
from src.data.loader import load_config, load_cnn_dailymail
from src.interpretability.attention_viz import (
    extract_seq2seq_attention,
    plot_attention_heatmap,
    plot_token_importance,
)

cfg = load_config("../config/config.yaml")
dataset = load_cnn_dailymail(cfg)

FIGURES_DIR = Path("../results/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Select a few diverse test examples
EXAMPLE_INDICES = [0, 10, 42]

## 1. Flan-T5-base — Cross-attention analysis

T5 is an encoder-decoder model. Its cross-attention shows, for each
generated summary token, which input (article) tokens the decoder
attended to. This is the most interpretable form of attention for
summarization.

In [ ]:
# Load best T5 checkpoint
# ADJUST path if a different V3 config won.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

BEST_T5_DIR = "v3_t5_A"  # <-- CHANGE if needed
t5_ckpt = Path(f"../results/checkpoints/{BEST_T5_DIR}")
t5_subdirs = sorted([p for p in t5_ckpt.iterdir() if p.name.startswith("checkpoint-")])
t5_path = t5_subdirs[-1] if t5_subdirs else t5_ckpt

t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_path, dtype=torch.float32)
t5_tok = AutoTokenizer.from_pretrained(t5_path, use_fast=True)
t5_model.to("cuda").eval()
print(f"T5 loaded from {t5_path}")

In [ ]:
# Extract and visualize attention for each example
for idx in EXAMPLE_INDICES:
    article = dataset["test"][idx]["article"]
    reference = dataset["test"][idx]["highlights"]

    print(f"\n{'='*70}")
    print(f"Example {idx}")
    print(f"Reference: {reference[:200]}")
    print(f"{'='*70}")

    attn_data = extract_seq2seq_attention(
        model=t5_model,
        tokenizer=t5_tok,
        article=article,
        max_input_length=cfg["models"]["t5"]["max_input_length"],
        max_new_tokens=64,
    )

    plot_attention_heatmap(
        attn_data,
        title=f"T5 cross-attention — Example {idx}",
        save_path=FIGURES_DIR / f"attn_t5_example_{idx}.png",
    )

    plot_token_importance(
        attn_data,
        title=f"T5 token importance — Example {idx}",
        save_path=FIGURES_DIR / f"token_importance_t5_{idx}.png",
    )

In [ ]:
# Free T5 VRAM
del t5_model
torch.cuda.empty_cache()

## 2. Qwen3-1.7B — Self-attention analysis

Qwen3 is a decoder-only model, so there is no cross-attention between
encoder and decoder. Instead, we analyze the self-attention patterns
in the last layer during generation — specifically, how the first
generated summary tokens attend back to the article tokens in the prompt.

This is less clean than T5's cross-attention but still reveals which
parts of the article the model focuses on.

In [ ]:
# Load best Qwen3 checkpoint
from peft import PeftModel
from src.models.loader import load_model

BEST_QWEN_DIR = "v3_qwen_A"  # <-- CHANGE if needed
qwen_base = load_model(cfg["models"]["qwen"])

qwen_ckpt = Path(f"../results/checkpoints/{BEST_QWEN_DIR}")
qwen_subdirs = sorted([p for p in qwen_ckpt.iterdir() if p.name.startswith("checkpoint-")])
qwen_path = qwen_subdirs[-1] if qwen_subdirs else qwen_ckpt

qwen_base.model = PeftModel.from_pretrained(qwen_base.model, str(qwen_path))
qwen_base.model.eval()
qwen_base.model.config.use_cache = True
print(f"Qwen3 loaded from {qwen_path}")

In [ ]:
# Extract self-attention from Qwen3 by running a forward pass on a
# prompt+summary sequence and inspecting the last layer's attention.
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

for idx in EXAMPLE_INDICES:
    article = dataset["test"][idx]["article"]
    reference = dataset["test"][idx]["highlights"]

    # Build the same prompt format used during training
    prompt = f"Summarize the following news article:\n\n{article}\n\nSummary:"

    inputs = qwen_base.tokenizer(
        prompt,
        max_length=512,  # keep manageable for attention extraction
        truncation=True,
        return_tensors="pt",
    ).to("cuda")

    # Forward pass with output_attentions to capture attention matrices
    with torch.no_grad():
        out = qwen_base.model(
            **inputs,
            output_attentions=True,
        )

    # Get last layer attention, average across heads
    # Shape: (batch, heads, seq_len, seq_len) → (seq_len, seq_len)
    last_layer_attn = out.attentions[-1][0].mean(dim=0).cpu().numpy()

    # We care about the last few tokens (near "Summary:") attending back
    # to the article tokens.
    input_ids = inputs["input_ids"][0].cpu().tolist()
    tokens = qwen_base.tokenizer.convert_ids_to_tokens(input_ids)

    # Show attention from the last 5 positions to the first 50 input tokens
    n_show_out = 5
    n_show_in = min(50, len(tokens))
    attn_slice = last_layer_attn[-n_show_out:, :n_show_in]
    in_tok = [t.replace("Ġ", " ").strip()[:12] for t in tokens[:n_show_in]]
    out_tok = [t.replace("Ġ", " ").strip()[:12] for t in tokens[-n_show_out:]]

    fig, ax = plt.subplots(figsize=(14, 3))
    sns.heatmap(attn_slice, xticklabels=in_tok, yticklabels=out_tok,
                cmap="YlOrRd", ax=ax)
    ax.set_title(f"Qwen3 self-attention (last layer, last 5 positions) — Example {idx}")
    ax.set_xlabel("Input tokens (article start)")
    plt.xticks(rotation=45, ha="right", fontsize=7)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"attn_qwen_example_{idx}.png", bbox_inches="tight", dpi=150)
    plt.show()

    print(f"Example {idx}: Reference = {reference[:150]}...\n")

In [ ]:
# Free Qwen VRAM
del qwen_base
torch.cuda.empty_cache()

## Análisis de interpretabilidad

*(Rellenar tras ejecución. Puntos a cubrir:)*

1. **¿Qué zonas del artículo atiende más cada modelo?** — principio, final,
   nombres propios, cifras, etc.
2. **¿Hay diferencias claras entre T5 y Qwen3?** — se espera que T5 con su
   cross-attention tenga patrones más concentrados, y Qwen3 más difusos.
3. **¿Los tokens más atendidos coinciden con lo que aparece en el resumen?**
   — si sí, el modelo está haciendo su trabajo; si no, indica extractividad
   vs abstractividad.
4. **Limitación:** la atención promediada entre capas/cabezas pierde
   especificidad. Cada cabeza puede especializarse en aspectos distintos
   (sintaxis, co-referencia, entidades). Un análisis por cabeza sería más
   fino pero excede el alcance de este trabajo.